# Day 5 — Merging, Exporting & Serving Fine-Tuned Models

---

You have a trained LoRA adapter. Now what? Today:

1. **Merge** the adapter into the base model (one big weight file)
2. **Export** to GGUF for local serving with **Ollama**
3. Deploy to a hosted endpoint on **Together AI** or **Hugging Face**


## 1. Adapter vs merged — which do you serve?

**Two choices at serve time:**

**a) Ship base + adapter separately.** The serving code loads the base model, then applies the adapter at startup. Small file transfer (adapter only ~50 MB), but the runtime needs to know how to merge.

**b) Merge weights and ship a single model.** One monolithic file. Simpler ops, larger download. This is what Ollama / vLLM / most hosted services expect.

**For freshers: merge, then ship.** Simpler.


## 2. Merge with `peft.merge_and_unload`


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"   # tiny (~135 MB); runs on CPU/MPS
OUT  = "triage-merged"

base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16)
tok  = AutoTokenizer.from_pretrained(BASE)

# Real use:  model = PeftModel.from_pretrained(base, "triage-lora")
# Here we attach a fresh (untrained) LoRA just to demonstrate merge mechanics.
lora = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base, lora)

merged = model.merge_and_unload()
merged.save_pretrained(OUT)
tok.save_pretrained(OUT)
print(f"merged model + tokenizer saved to ./{OUT}")


## 3. Export to GGUF for Ollama

**GGUF** is a single-file quantized format used by `llama.cpp`, Ollama, LM Studio, and most local LLM tools. It runs on CPU + GPU on any laptop.

Unsloth ships a one-liner for GGUF export:


In [ ]:
# Runs on macOS/Linux — pure-Python converter, no CUDA needed.
# Unsloth's `save_pretrained_gguf` is the one-liner for GPU/Colab; on a Mac we
# use llama.cpp's converter directly.
import subprocess, sys
from pathlib import Path

if not Path("llama.cpp").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/ggerganov/llama.cpp",
    ])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "llama.cpp/requirements/requirements-convert_hf_to_gguf.txt",
])

subprocess.check_call([
    sys.executable, "llama.cpp/convert_hf_to_gguf.py",
    "triage-merged",
    "--outfile", "triage-f16.gguf",
    "--outtype", "f16",
])
print("wrote triage-f16.gguf")

# To quantize to q4_k_m you need llama.cpp's `quantize` binary:
#   cd llama.cpp && cmake -B build && cmake --build build --target llama-quantize
#   ./build/bin/llama-quantize ../triage-f16.gguf ../triage-Q4_K_M.gguf Q4_K_M
# The f16 file works fine with Ollama for a 135M model; quantize only for larger ones.


**Quantization choices** (the trade-off you'll see everywhere):

| Method | Size | Quality | When |
|---|---|---|---|
| `q4_k_m` | Small | Great | Default for CPU serving |
| `q5_k_m` | Medium | Better | If you have RAM to spare |
| `q8_0` | Large | Near-lossless | Comparing merged vs base |
| `f16` | Full | Reference | Debugging only |

`q4_k_m` is the standard default in 2026 for local serving.


## 4. Run it in Ollama (on your laptop)

Once you have the `.gguf` file:

```bash
# 1. Install ollama - https://ollama.com
brew install ollama         # macOS. Similar one-liner on Linux/Win.

# 2. Create a Modelfile
cat > Modelfile <<'EOF'
FROM ./triage-Q4_K_M.gguf
TEMPLATE "{{ .System }}\n{{ .Prompt }}"
PARAMETER temperature 0
EOF

# 3. Register the model
ollama create triage -f Modelfile

# 4. Chat
ollama run triage "My card was charged twice."
```

Ollama exposes a local HTTP API on `http://localhost:11434` — same shape as OpenAI. Wire it into any of your Section 6 / 7 code by swapping the client.


## 5. When to go hosted instead — vLLM & Together AI

**vLLM** is a high-throughput inference server. If you're serving your fine-tune to many concurrent users, `vLLM` is much faster than Ollama or plain `transformers`.

```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server \
       --model ./triage-merged --host 0.0.0.0 --port 8000
```

Also exposes an OpenAI-compatible API. Requires a real GPU.

**Together AI hosted fine-tuning** — for small teams: skip serving entirely. Upload your JSONL, they fine-tune LLaMA-3 and give you an OpenAI-shaped endpoint. Costs a few dollars per training run + per-token inference. See docs at https://docs.together.ai/docs/fine-tuning-overview.


## 6. Serving decision cheat sheet

| Scenario | Serve with |
|---|---|
| Prototype on your laptop | **Ollama** |
| Single machine, moderate traffic | **vLLM** on your own GPU box |
| Multi-user, don't want ops | **Together AI hosted fine-tune** or Hugging Face Inference Endpoints |
| iOS / edge | Convert to **CoreML** or **MLC-LLM** (out of scope here) |


## 7. Hugging Face Hub — the model registry step

Push your merged model or adapter to the HF Hub so you (or your team) can pull it anywhere.

```python
model.push_to_hub("your-username/triage-llama-3.2-3b-lora")
tokenizer.push_to_hub("your-username/triage-llama-3.2-3b-lora")
```

Set `HF_TOKEN` in your env, get one at https://huggingface.co/settings/tokens.

This is your **version control for models.** Every experiment gets its own repo tag — you can go back to any prior version.


## Recap

- **Merge** the LoRA adapter into the base for simpler serving (`peft.merge_and_unload`).
- **Export to GGUF** with `save_pretrained_gguf` (via Unsloth) for CPU-friendly local serving.
- **Ollama** = your laptop's LLM server. Register a `.gguf` and go.
- **vLLM** for high-throughput self-hosted serving. **Together AI hosted** if you'd rather not run infra.
- **Push to Hugging Face Hub** — version control for models.
- **Next class:** the capstone — a real domain-specific fine-tune, end to end.
